# Predict Workout

## Import Libraries

In [25]:
import pickle
import pandas as pd
import numpy as np
from IPython.display import display

## Load Model

In [26]:
# ==============================================================================
# 1. MODEL & RESOURCE LOADING
# ==============================================================================
# Objective: Load the trained workout recommendation system.
#
# The 'model_workout.pickle' file contains:
# - knn_model: The core Nearest Neighbor algorithm.
# - scaler: To normalize user inputs (Age, Weight, etc.) to 0-1 scale.
# - weights: Custom importance weights for each feature (e.g., Goal > Age).
# - profiles_db: The database of existing user profiles (reference data).
# - schedule_db: The database of workout routines linked to profiles.
# - encoders: Tools to convert text (Male, Gym) to numbers.
# - features: The specific column order required by the model.

path = '../../models/model_workout.pickle'

try:
    with open(path, 'rb') as f:
        model_data = pickle.load(f)
        
    knn = model_data['knn_model']
    scaler = model_data['scaler']
    weights = model_data['weights']
    db_profiles = model_data['profiles_db']
    db_schedule = model_data['schedule_db']
    encoders = model_data['encoders']
    feature_order = model_data['features'] 

    print("Workout Model Loaded & Ready.")

except FileNotFoundError:
    print(f"Error: Model file not found at {path}")
    exit()

Workout Model Loaded & Ready.


## Predict Function

In [27]:
# ==============================================================================
# 2. PREDICTION LOGIC (FUNCTION)
# ==============================================================================

def predict_workout_plan(user_input):
    """
    Generates a personalized workout schedule by finding a 'Look-alike' user.

    This function takes a new user's profile, encodes it, normalizes it, 
    and uses a Weighted KNN search to find the most similar existing user 
    in the database. It then returns that user's proven workout schedule.

    ---------------------------------------------------------------------------
    Args:
        user_input (dict): A dictionary containing all user attributes.
                           Required keys: Age, Gender, Height, Weight, Goal, 
                           Level, Environment, Frequency, Duration, and Sports flags.

    Returns:
        pd.DataFrame: A DataFrame containing the full weekly schedule of the 
                      matched user, sorted by Day. Returns None if validation fails.
    """
    
    # --- STEP 1: ENCODE INPUT (Text -> Number) ---
    # Objective: Convert categorical strings into numerical format using 
    # the same encoders used during training.
    try:
        # Extract environment preference
        env_input = user_input['Environment'] 
        
        # Transform categories using pre-loaded LabelEncoders
        goal_enc = encoders['goal'].transform([user_input['Goal']])[0]
        level_enc = encoders['level'].transform([user_input['level']])[0]
        gender_enc = encoders['gender'].transform([user_input['Gender']])[0]
        env_enc = encoders['environment'].transform([env_input])[0] 
        
    except KeyError as e:
        print(f"Error: Missing required input field: {e}")
        return None
    except ValueError as e:
        print(f"Error: Invalid category value (e.g., Typo in 'Gym' vs 'Gim'). Details: {e}")
        return None

    # --- STEP 2: DATA MAPPING (Construct Feature Vector) ---
    # Objective: Organize the input data into a dictionary with keys matching 
    # the training data columns.
    input_data = {
        # Physical Attributes
        'Age_x': user_input['Age'],
        'Gender_Encoded': gender_enc,
        'Height_cm_x': user_input['Height_cm'],
        'Initial_Weight_kg_x': user_input['Weight_kg'],
        'Body_Fat_Category_x': user_input['Body_Fat_Category'],
        'Body_Fat_Percentage_x': user_input['Body_Fat_Percentage_x'],
        
        # Workout Preferences
        'Goal_Encoded': goal_enc,
        'Workout_Frequency_x': user_input['Workout_Frequency'],
        'Average_Duration_Minutes_x': user_input['Average_Duration_Minutes'],
        'level_Encoded': level_enc,
        'environment_Encoded': env_enc,
        
        # Sports/Hobbies (Binary Flags: 0 or 1)
        'Badminton': user_input['Badminton'],
        'Football': user_input['Football'],
        'Basketball': user_input['Basketball'],
        'Volleyball': user_input['Volleyball'],
        'Swim': user_input['Swim']
    }
    
    # --- STEP 3: DATAFRAME CONSTRUCTION ---
    # Objective: Convert the dictionary to a DataFrame and enforce strict column ordering.
    try:
        # Create a single-row DataFrame
        input_df = pd.DataFrame([input_data])
        
        # Reorder columns to match 'feature_order' (Critical for Scikit-Learn models)
        input_df = input_df[feature_order] 
        
    except KeyError as e:
        print(f"Error: Model expects column {e}, but it's missing in our input map.")
        return None
    
    # --- STEP 4: NEAREST NEIGHBOR SEARCH (KNN) ---
    # Objective: Find the closest match in the vector space.
    
    # 1. Normalize (Scale to 0-1)
    # Allows fair comparison between different units (e.g., Age vs Weight).
    input_scaled = scaler.transform(input_df)
    
    # 2. Apply Custom Weights
    # Increases the importance of specific features (e.g., Goal matches are more important than Age matches).
    input_weighted = pd.DataFrame(input_scaled, columns=feature_order)
    for col, weight in weights.items():
        if col in input_weighted.columns:
            input_weighted[col] = input_weighted[col] * weight
            
    # 3. Query the Model (k=1)
    # Find the index of the single most similar user profile.
    distances, indices = knn.kneighbors(input_weighted.values, n_neighbors=1)
    
    matched_index = indices[0][0]
    matched_user = db_profiles.iloc[matched_index]
    matched_user_id = matched_user['User_ID']
    
    print("\n" + "="*40)
    print(f"SEARCH RESULT (MATCHING PROFILE)")
    print("-" * 40)
    print(f"Matched User ID : {matched_user_id}")
    print(f"   • User Env Pref : {user_input['Environment']}")
    print(f"   • Match Env Pref: {matched_user['Environment']}") 
    print("="*40)
    
    # --- STEP 5: RETRIEVE SCHEDULE ---
    # Objective: Fetch the workout plan associated with the matched User ID from the schedule database.
    schedule = db_schedule[db_schedule['User_ID'] == matched_user_id].copy()
    
    # Sort by Day for logical display order
    if 'Day' in schedule.columns:
        schedule = schedule.sort_values('Day')
        
    return schedule


## Execution

In [28]:
# ==============================================================================
# 3. TEST EXECUTION (MAIN BLOCK)
# ==============================================================================

# 1. Define Test Input (Mock User)
input_user = {
    'Age': 25, 
    'Gender': 'Male', 
    'Height_cm': 175, 
    'Weight_kg': 60,
    'Body_Fat_Category': 2, 
    'Body_Fat_Percentage_x': 15.0,
    'Goal': 'Muscle Gain', 
    'Workout_Frequency': 4, 
    'Average_Duration_Minutes': 60,
    'level': 'Beginner',
    
    'Environment': 'Home', 
    
    # Sports Flags
    'Badminton': 0, 
    'Football': 1, 
    'Basketball': 0, 
    'Volleyball': 0, 
    'Swim': 1
}

# 2. Run Prediction
plan = predict_workout_plan(input_user)

# 3. Display Results
if plan is not None:
    # Select relevant columns for clear display
    cols = ['Day', 'Muscle Group', 'Exercise Name', 'Sets', 'Reps', 'Instructions', 'Workout_Type', 'Calories_Burned', 'Duration_Minutes', 'Rest_Minutes']
    
    # Iterate through unique days and print the schedule
    for day in plan['Day'].unique():
        print(f"\n{day}")
        # Reset index for cleaner table view
        display(plan[plan['Day'] == day][cols].reset_index(drop=True))


SEARCH RESULT (MATCHING PROFILE)
----------------------------------------
Matched User ID : 187
   • User Env Pref : Home
   • Match Env Pref: Home

Day 1 - Upper Strength


,Day,Muscle Group,Exercise Name,Sets,Reps,Instructions,Workout_Type,Calories_Burned,Duration_Minutes,Rest_Minutes
0,Day 1 - Upper Strength,Chest,decline push up,4,8-12,"Step:1 Feet on a chair, hands on floor., Step:...",Strength,50,4,8
1,Day 1 - Upper Strength,Chest,diamond push up,4,8-12,Step:1 Hands close together forming a diamond....,Strength,50,4,8
2,Day 1 - Upper Strength,Back,superman,4,8-12,"Step:1 Lie face down on the floor., Step:2 Lif...",Strength,50,4,8
3,Day 1 - Upper Strength,Shoulders,inchworm,4,8-12,"Step:1 Stand tall., Step:2 Walk hands out to p...",Strength,50,4,8
4,Day 1 - Upper Strength,Abs,side plank,4,8-12,"Step:1 Prop yourself on one elbow., Step:2 Lif...",Strength,50,4,8



Day 2 - Lower Quads


,Day,Muscle Group,Exercise Name,Sets,Reps,Instructions,Workout_Type,Calories_Burned,Duration_Minutes,Rest_Minutes
0,Day 2 - Lower Quads,Abs,flutter kicks,4,8-12,"Step:1 Lie on back., Step:2 Kick legs up and d...",Strength,50,4,8
1,Day 2 - Lower Quads,Calves,standing calf raise,4,8-12,"Step:1 Stand on the edge of a step., Step:2 Lo...",Strength,50,4,8
2,Day 2 - Lower Quads,Cardio,High Knees,1,13 Mins,"Step:1 Run in place., Step:2 Lift your knees a...",Cardio,65,13,0
3,Day 2 - Lower Quads,Quads,reverse lunge,4,8-12,"Step:1 Stand tall., Step:2 Step one foot back ...",Strength,50,4,8
4,Day 2 - Lower Quads,Quads,bulgarian split squat,4,8-12,"Step:1 One foot on chair behind you., Step:2 S...",Strength,50,4,8



Day 3 - Upper Pump


,Day,Muscle Group,Exercise Name,Sets,Reps,Instructions,Workout_Type,Calories_Burned,Duration_Minutes,Rest_Minutes
0,Day 3 - Upper Pump,Biceps,superman,4,8-12,"Step:1 Lie face down on the floor., Step:2 Lif...",Strength,50,4,8
1,Day 3 - Upper Pump,Triceps,impossible dips,4,8-12,Step:1 Position yourself between two parallel ...,Strength,50,4,8
2,Day 3 - Upper Pump,Chest,diamond push up,4,8-12,Step:1 Hands close together forming a diamond....,Strength,50,4,8



Day 4 - Lower Hams


,Day,Muscle Group,Exercise Name,Sets,Reps,Instructions,Workout_Type,Calories_Burned,Duration_Minutes,Rest_Minutes
0,Day 4 - Lower Hams,Glutes,bent knee lying twist (male),4,8-12,Step:1 Lie flat on your back with your knees b...,Strength,50,4,8
1,Day 4 - Lower Hams,Hamstrings,single leg deadlift,4,8-12,"Step:1 Stand on one leg., Step:2 Hinge at your...",Strength,50,4,8
2,Day 4 - Lower Hams,Hamstrings,glute bridge,4,8-12,"Step:1 Lie on your back with knees bent., Step...",Strength,50,4,8
3,Day 4 - Lower Hams,Glutes,single leg glute bridge,4,8-12,"Step:1 Lie on back, one knee bent, other leg s...",Strength,50,4,8
4,Day 4 - Lower Hams,Calves,standing calf raise,4,8-12,"Step:1 Stand on the edge of a step., Step:2 Lo...",Strength,50,4,8
